# TUGAS MANDIRI — Pertemuan 12
## Klasifikasi & ML Pipeline dengan Spark MLlib

| | |
|---|---|
| **Nama** | Angger Naufal Abror |
| **NPM** | 2505060020 |
| **Mata Kuliah** | Praktikum Big Data |
| **Program Studi** | Teknologi Informasi — Universitas Tidar |

### Skenario
"Kopi Nusantara" ingin membangun model untuk memprediksi apakah seorang pelanggan akan memberikan **ulasan positif (rating ≥ 4) atau tidak** berdasarkan atribut pesanannya, agar kombinasi menu/cabang yang berisiko mengecewakan pelanggan dapat diidentifikasi lebih awal.

---
## Persiapan: Membuat Dataset

Cell berikut berasal dari modul dan membuat file `tugas12_ulasan.csv`.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(212)
n = 3000
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas12 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
})

# Simulasi: waktu tunggu lebih lama cenderung (namun tidak selalu) menurunkan peluang ulasan positif
prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)

data_tugas12.to_csv("tugas12_ulasan.csv", index=False)
print(f"Dataset berhasil dibuat: {len(data_tugas12)} baris")
print("Distribusi label:")
print(data_tugas12["ulasan_positif"].value_counts())

Dataset berhasil dibuat: 3000 baris
Distribusi label:
ulasan_positif
0    1610
1    1390
Name: count, dtype: int64


## Membuat SparkSession & Membaca CSV

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Tugas12-Klasifikasi-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

26/11/19 07:46:07 WARN Utils: Your hostname, angger resolves to a loopback address: 127.0.1.1; using 10.90.65.21 instead (on interface wlp3s0)
26/11/19 07:46:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 07:46:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [2]:
df = spark.read.csv("tugas12_ulasan.csv", header=True, inferSchema=True)

df.printSchema()
df.show(5)
print("Jumlah baris:", df.count())
print("Distribusi label:")
df.groupBy("ulasan_positif").count().orderBy("ulasan_positif").show()

root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- waktu_tunggu_menit: integer (nullable = true)
 |-- ulasan_positif: integer (nullable = true)

+--------------+----------+-----------+---+------------------+--------------+
|     nama_menu|    cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|
+--------------+----------+-----------+---+------------------+--------------+
|     Kopi Susu|Yogyakarta|       Sore|  5|                 5|             1|
|        Es Teh|Yogyakarta|       Pagi|  1|                21|             0|
|    Cappuccino|Yogyakarta|       Sore|  5|                15|             0|
|Kentang Goreng|  Semarang|      Malam|  2|                 5|             1|
|     Americano|Yogyakarta|      Malam|  2|                10|             0|
+--------------+----------+-----------+---+------------------+--------------+
only showing top 5 rows

Jumlah bari

---
## A. Membangun ML Pipeline *(bobot 25%)*

Urutan tahap pipeline:
1. **`StringIndexer`** untuk `nama_menu`, `cabang`, dan `waktu_pesan`: mengubah teks kategori menjadi indeks angka.
2. **`OneHotEncoder`** untuk ketiga kolom indeks: mengubah indeks menjadi vektor biner agar model tidak menganggap indeks sebagai urutan besar-kecil.
3. **`VectorAssembler`**: menggabungkan hasil one-hot encoding dengan `qty` dan `waktu_tunggu_menit` menjadi satu kolom `features`.
4. **`LogisticRegression`** dengan `labelCol="ulasan_positif"`.

In [3]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression

# Tahap 1-3: StringIndexer
idx_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx")

# Tahap 4: OneHotEncoder untuk ketiga kolom indeks
ohe = OneHotEncoder(
    inputCols=["menu_idx", "cabang_idx", "waktu_idx"],
    outputCols=["menu_vec", "cabang_vec", "waktu_vec"]
)

# Tahap 5: VectorAssembler
assembler = VectorAssembler(
    inputCols=["menu_vec", "cabang_vec", "waktu_vec", "qty", "waktu_tunggu_menit"],
    outputCol="features"
)

# Tahap 6: Logistic Regression
lr = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")

# Lima tahap feature engineering disimpan dalam satu list agar dapat dipakai ulang di bagian C
feature_stages = [idx_menu, idx_cabang, idx_waktu, ohe, assembler]
pipeline_lr = Pipeline(stages=feature_stages + [lr])

print("Tahapan pipeline:")
for i, stage in enumerate(pipeline_lr.getStages(), start=1):
    print(f"  {i}. {type(stage).__name__}")

Tahapan pipeline:
  1. StringIndexer
  2. StringIndexer
  3. StringIndexer
  4. OneHotEncoder
  5. VectorAssembler
  6. LogisticRegression


---
## B. Pelatihan & Evaluasi *(bobot 25%)*

Data dibagi 80:20 dengan `seed=77`. Pipeline dilatih pada data training, lalu dievaluasi pada data testing dengan **accuracy**, **F1-score**, dan **AUC-ROC**.

In [4]:
train, test = df.randomSplit([0.8, 0.2], seed=77)
print(f"Data training: {train.count()} | Data testing: {test.count()}")

model_lr = pipeline_lr.fit(train)
predictions_lr = model_lr.transform(test)

predictions_lr.select(
    "nama_menu", "cabang", "waktu_pesan", "qty", "waktu_tunggu_menit",
    "ulasan_positif", "prediction", "probability"
).show(10, truncate=False)

Data training: 2411 | Data testing: 589
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|nama_menu|cabang  |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|probability                             |
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|Americano|Magelang|Malam      |1  |9                 |1             |1.0       |[0.30580926470532666,0.6941907352946733]|
|Americano|Magelang|Malam      |1  |15                |1             |0.0       |[0.6253879175414343,0.3746120824585657] |
|Americano|Magelang|Malam      |2  |2                 |1             |1.0       |[0.08601505227726715,0.9139849477227329]|
|Americano|Magelang|Malam      |3  |4                 |1             |1.0       |[0.1291608175231565,0.8708391824768436] |
|Americano|Magelang|Malam      |3  |13                |0             |0.0       |[0.522484905890236

In [5]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

ev_acc = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
ev_f1 = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
ev_auc = BinaryClassificationEvaluator(labelCol="ulasan_positif", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

acc_lr = ev_acc.evaluate(predictions_lr)
f1_lr = ev_f1.evaluate(predictions_lr)
auc_lr = ev_auc.evaluate(predictions_lr)

print("=== Evaluasi Logistic Regression ===")
print(f"Accuracy : {acc_lr:.4f}")
print(f"F1-score : {f1_lr:.4f}")
print(f"AUC-ROC  : {auc_lr:.4f}")

=== Evaluasi Logistic Regression ===
Accuracy : 0.7929
F1-score : 0.7926
AUC-ROC  : 0.8650


### Confusion Matrix (Logistic Regression)

In [6]:
predictions_lr.groupBy("ulasan_positif", "prediction").count() \
    .orderBy("ulasan_positif", "prediction").show()

# Interpretasi:
# ulasan_positif=0, prediction=0.0 -> True Negative  (benar memprediksi ulasan tidak positif)
# ulasan_positif=0, prediction=1.0 -> False Positive (diprediksi positif, padahal tidak)
# ulasan_positif=1, prediction=0.0 -> False Negative (diprediksi tidak positif, padahal positif)
# ulasan_positif=1, prediction=1.0 -> True Positive  (benar memprediksi ulasan positif)

+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



---
## C. Model Pembanding: Random Forest *(bobot 20%)*

Lima tahap feature engineering dari bagian A (`feature_stages`) dipakai ulang tanpa perubahan. Hanya tahap terakhir yang diganti menjadi `RandomForestClassifier` dengan `numTrees=30`. Parameter `seed=42` ditambahkan agar hasil Random Forest tetap sama setiap kali dijalankan.

In [7]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(featuresCol="features", labelCol="ulasan_positif", numTrees=30, seed=42)
pipeline_rf = Pipeline(stages=feature_stages + [rf])

model_rf = pipeline_rf.fit(train)
predictions_rf = model_rf.transform(test)

acc_rf = ev_acc.evaluate(predictions_rf)
f1_rf = ev_f1.evaluate(predictions_rf)
auc_rf = ev_auc.evaluate(predictions_rf)

print("Confusion matrix Random Forest:")
predictions_rf.groupBy("ulasan_positif", "prediction").count() \
    .orderBy("ulasan_positif", "prediction").show()

Confusion matrix Random Forest:
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  261|
|             0|       1.0|   53|
|             1|       0.0|   67|
|             1|       1.0|  208|
+--------------+----------+-----+



In [14]:
import pandas as pd

perbandingan = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest (30 trees)"],
    "Accuracy": [acc_lr, acc_rf],
    "F1-score": [f1_lr, f1_rf],
    "AUC-ROC": [auc_lr, auc_rf],
}).round(4)
print(perbandingan.to_string(index=False))

# Model terbaik dipilih berdasarkan AUC-ROC tertinggi (sesuai instruksi bagian D)
if auc_lr >= auc_rf:
    best_name, best_model, best_predictions = "Logistic Regression", model_lr, predictions_lr
else:
    best_name, best_model, best_predictions = "Random Forest", model_rf, predictions_rf
print(f"\nModel terbaik berdasarkan AUC-ROC: {best_name}")

                   Model  Accuracy  F1-score  AUC-ROC
     Logistic Regression    0.7929    0.7926   0.8650
Random Forest (30 trees)    0.7963    0.7958   0.8474

Model terbaik berdasarkan AUC-ROC: Logistic Regression


**Analisis perbandingan:**

- **Accuracy dan F1-score** kedua model hampir sama (sekitar 0,79–0,80). Random Forest hanya sedikit lebih tinggi, selisihnya sekitar 2 prediksi benar dari ±589 data testing, jadi perbedaannya tidak berarti.
- **AUC-ROC Logistic Regression lebih tinggi** (sekitar 0,865 dibanding 0,847). Artinya Logistic Regression lebih baik dalam **mengurutkan peluang** pelanggan memberi ulasan positif di berbagai ambang batas, bukan hanya pada ambang 0,5.
- Hal ini masuk akal karena pola data dibuat dengan fungsi **logistik (sigmoid)** dari `waktu_tunggu_menit`, bentuk yang memang dimodelkan oleh Logistic Regression. Random Forest memecah fitur menjadi potongan-potongan bertingkat sehingga peluangnya lebih kasar.
- Kesimpulannya, model yang lebih kompleks belum tentu lebih baik. **Logistic Regression dipilih sebagai model terbaik** karena AUC-ROC-nya tertinggi, lebih sederhana, dan koefisiennya mudah ditafsirkan.

Dibantu Gemini Untuk menganalisis perbandingan ini


---
## D. Analisis Kesalahan Model *(bobot 15%)*

Menampilkan 10 baris yang **salah diprediksi** oleh model terbaik (`ulasan_positif != prediction`), lalu mencari pola kesalahannya.

In [15]:
salah = best_predictions.filter(col("ulasan_positif") != col("prediction"))

print(f"Model: {best_name}")
print(f"Jumlah prediksi salah: {salah.count()} dari {best_predictions.count()} data testing")

salah.select(
    "nama_menu", "cabang", "waktu_pesan", "qty", "waktu_tunggu_menit",
    "ulasan_positif", "prediction", "probability"
).show(10, truncate=False)

Model: Logistic Regression
Jumlah prediksi salah: 122 dari 589 data testing
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|nama_menu|cabang  |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|probability                             |
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|Americano|Magelang|Malam      |1  |15                |1             |0.0       |[0.6253879175414343,0.3746120824585657] |
|Americano|Magelang|Siang      |2  |22                |1             |0.0       |[0.9113822566149014,0.08861774338509865]|
|Americano|Magelang|Sore       |1  |7                 |0             |1.0       |[0.23209137939008387,0.7679086206099162]|
|Americano|Magelang|Sore       |2  |11                |0             |1.0       |[0.42615454016905935,0.5738454598309406]|
|Americano|Magelang|Sore       |3  |5                 |0       

In [16]:
from pyspark.sql.functions import avg, count, round as spark_round

# Tingkat kesalahan per nilai waktu tunggu: apakah kesalahan menumpuk di sekitar ambang tertentu?
best_predictions \
    .withColumn("salah", (col("ulasan_positif") != col("prediction")).cast("int")) \
    .groupBy("waktu_tunggu_menit") \
    .agg(
        count("*").alias("jumlah_data"),
        spark_round(avg("ulasan_positif"), 2).alias("proporsi_positif_aktual"),
        spark_round(avg("salah"), 2).alias("tingkat_kesalahan")
    ) \
    .orderBy("waktu_tunggu_menit") \
    .show(30)

+------------------+-----------+-----------------------+-----------------+
|waktu_tunggu_menit|jumlah_data|proporsi_positif_aktual|tingkat_kesalahan|
+------------------+-----------+-----------------------+-----------------+
|                 2|         25|                   0.96|             0.04|
|                 3|         21|                   0.86|             0.14|
|                 4|         35|                   0.94|             0.06|
|                 5|         22|                   0.91|             0.09|
|                 6|         31|                   0.81|             0.19|
|                 7|         26|                   0.81|             0.19|
|                 8|         28|                   0.75|             0.25|
|                 9|         28|                   0.79|             0.21|
|                10|         23|                   0.65|             0.35|
|                11|         19|                   0.42|             0.58|
|                12|     

In [17]:
# Jenis kesalahan: False Positive vs False Negative
salah.groupBy("ulasan_positif", "prediction").count().show()

print("Rata-rata waktu tunggu pada baris yang salah:")
salah.select(spark_round(avg("waktu_tunggu_menit"), 2).alias("rata2_waktu_tunggu_salah")).show()

+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             1|       0.0|   65|
|             0|       1.0|   57|
+--------------+----------+-----+

Rata-rata waktu tunggu pada baris yang salah:
+------------------------+
|rata2_waktu_tunggu_salah|
+------------------------+
|                   12.65|
+------------------------+



In [18]:
# Koefisien Logistic Regression: fitur mana yang paling berpengaruh?
lr_model = model_lr.stages[-1]
attrs = predictions_lr.schema["features"].metadata["ml_attr"]["attrs"]
nama_fitur = sorted([a for grup in attrs.values() for a in grup], key=lambda a: a["idx"])

koef = pd.DataFrame({
    "fitur": [a["name"] for a in nama_fitur],
    "koefisien": lr_model.coefficients.toArray(),
    "importance_RF": model_rf.stages[-1].featureImportances.toArray(),
})
koef["|koefisien|"] = koef["koefisien"].abs()
print(koef.sort_values("importance_RF", ascending=False).round(4).to_string(index=False))

                fitur  koefisien  importance_RF  |koefisien|
   waktu_tunggu_menit    -0.2220         0.9430       0.2220
                  qty    -0.0108         0.0113       0.0108
   menu_vec_Kopi Susu     0.1171         0.0094       0.1171
      waktu_vec_Malam     0.2530         0.0059       0.2530
cabang_vec_Yogyakarta     0.0645         0.0058       0.0645
  cabang_vec_Semarang    -0.1140         0.0049       0.1140
       waktu_vec_Sore     0.1857         0.0043       0.1857
       waktu_vec_Pagi    -0.0651         0.0041       0.0651
  menu_vec_Roti Bakar    -0.1443         0.0041       0.1443
   menu_vec_Americano    -0.0855         0.0033       0.0855
      menu_vec_Es Teh    -0.0191         0.0024       0.0191
  menu_vec_Cappuccino     0.0194         0.0015       0.0194


---
## E. Kesimpulan Bisnis *(bobot 15%)*

Berdasarkan model klasifikasi yang dibangun, **waktu tunggu adalah faktor paling menentukan** kepuasan pelanggan Kopi Nusantara. Koefisien Logistic Regression untuk `waktu_tunggu_menit` bernilai negatif (sekitar −0,22 per menit), dan Random Forest memberi bobot kepentingan sekitar 94% pada fitur ini. Sebaliknya, jenis menu, cabang, waktu pemesanan, dan jumlah pesanan hampir tidak memengaruhi prediksi. Jadi masalah kepuasan pelanggan bukan terletak pada produk atau lokasi tertentu, melainkan pada **kecepatan layanan**.

Rekomendasi konkret untuk manajemen:

1. Tetapkan standar layanan maksimal 10 menit. Analisis kesalahan menunjukkan bahwa di bawah 10 menit mayoritas pelanggan memberi ulasan positif, sedangkan sekitar 12 menit peluangnya turun ke 50:50, dan di atas 17 menit ulasan positif sudah jarang.
2. Pantau waktu tunggu secara real-time di setiap cabang. Pesanan yang mendekati 10 menit diberi prioritas atau ditangani barista tambahan.
3. Tambah staf dan siapkan bahan lebih awal pada jam sibuk untuk menekan antrean.
4. Lakukan pemulihan layanan untuk pesanan yang terlanjur melewati ±12 menit, misalnya permintaan maaf, voucher, atau bonus kecil, karena pelanggan di zona ini paling mudah berubah dari puas menjadi kecewa.
5. Pakai model sebagai sistem peringatan dini. Model dapat memberi skor risiko ulasan negatif per pesanan sehingga tim bisa bertindak sebelum pelanggan memberi ulasan.

Catatan: dataset ini hasil simulasi, sehingga sebelum diterapkan, model sebaiknya divalidasi ulang dengan data ulasan asli Kopi Nusantara.

---
## Menutup SparkSession

In [ ]:
spark.stop()
print("SparkSession ditutup.")